# My Selective Surveillance Scenario Test

I use this notebook to record how the same system behaves in different conditions. I do not compare different models here. I compare normal light, darkness, hats, twins, printed photos, and video replays.

**Important:** this notebook calculates results only from trials I enter. It does not create test results or prove that the current system can detect a photo or screen. The current project has no liveness or anti-spoofing model. Test only with consenting participants and harmless, controlled demonstrations.

## 1. What I record for each trial

Each row is one controlled trial. I record whether a person should enter the zone and whether the system detected that entry. For face trials, `expected_identity` is the enrolled label or `Unknown`; `predicted_identity` is the dashboard result, or `Uncertain`/`NotChecked`. I record photo or screen attacks with `presentation_attack` and whether the system accepted the displayed identity in `attack_accepted`. Times are seconds. Never put a participant's real name or image in a notebook committed to Git.

In [ ]:
# I import only Python's built-in tools for this evaluation.
from collections import defaultdict
from statistics import mean

# I replace these example rows with my own measured trials.
# Keep one row per controlled trial. Do not make up values.
trials = [
    # {"scenario": "normal_light", "expected_identity": "Person_A",
    #  "predicted_identity": "Person_A", "expected_zone_entry": True,
    #  "detected_zone_entry": True, "face_checked": True,
    #  "alerted": False,
    #  "presentation_attack": "none", "attack_accepted": None,
    #  "entry_to_alert_seconds": None, "fps": 12.4},
]

print(f"Trials entered: {len(trials)}")

## 2. Check my trial records

This cell catches missing labels and invalid values before I calculate rates.

In [ ]:
required_fields = {"scenario", "expected_identity", "predicted_identity",
                  "expected_zone_entry", "detected_zone_entry", "alerted"}
for row_number, trial in enumerate(trials, start=1):
    missing = required_fields - trial.keys()
    if missing:
        raise ValueError(f"Trial {row_number} is missing: {sorted(missing)}")
    if trial["expected_identity"] == "" or trial["predicted_identity"] == "":
        raise ValueError(f"Trial {row_number} needs identity labels. Use 'Unknown', 'Uncertain' or 'NotChecked' where appropriate.")
    for field in ("expected_zone_entry", "detected_zone_entry", "alerted"):
        if not isinstance(trial[field], bool):
            raise ValueError(f"Trial {row_number}: {field} must be True or False.")
    if "face_checked" in trial and not isinstance(trial["face_checked"], bool):
        raise ValueError(f"Trial {row_number}: face_checked must be True or False.")

if not trials:
    print("No results yet. Add measured trial rows in the cell above.")
else:
    print(f"I checked {len(trials)} trial record(s).")

## 3. Calculate recognition and alert measures

I calculate face recognition only for trials where a face was actually checked. I report missed zone entries and false zone detections separately from identity errors. For printed photos, screens and replays, I report whether an identity was accepted, but this is only an observed presentation-attack acceptance rate; the current system has no liveness detector. A rate is `Not measured` when its denominator is zero.

In [ ]:
def percentage(numerator, denominator):
    if denominator == 0:
        return "Not measured"
    return round(100 * numerator / denominator, 2)

def summarize(rows):
    checked = [r for r in rows if r.get("face_checked", True)]
    enrolled = [r for r in checked if r["expected_identity"] != "Unknown"]
    unenrolled = [r for r in checked if r["expected_identity"] == "Unknown"]
    correct = sum(r["predicted_identity"] == r["expected_identity"] for r in enrolled)
    false_rejects = sum(r["predicted_identity"] in ("Unknown", "Uncertain", "NotChecked") for r in enrolled)
    wrong_identity = sum(r["predicted_identity"] not in (r["expected_identity"], "Unknown", "Uncertain", "NotChecked") for r in enrolled)
    false_matches = sum(r["predicted_identity"] not in ("Unknown", "Uncertain", "NotChecked") for r in unenrolled)
    expected_entries = [r for r in rows if r["expected_zone_entry"]]
    expected_non_entries = [r for r in rows if not r["expected_zone_entry"]]
    detected_entries = sum(r["detected_zone_entry"] for r in expected_entries)
    false_zone_entries = sum(r["detected_zone_entry"] for r in expected_non_entries)
    attack_trials = [r for r in rows if r.get("presentation_attack", "none") != "none" and r.get("attack_accepted") is not None]
    accepted_attacks = sum(bool(r["attack_accepted"]) for r in attack_trials)
    alerts = sum(bool(r["alerted"]) for r in rows)
    latencies = [r["entry_to_alert_seconds"] for r in rows
                 if r.get("entry_to_alert_seconds") is not None]
    frame_rates = [r["fps"] for r in rows if r.get("fps") is not None]
    return {
        "trials": len(rows),
        "known_person_correct_identity_percent": percentage(correct, len(enrolled)),
        "known_person_false_reject_percent": percentage(false_rejects, len(enrolled)),
        "known_person_wrong_identity_percent": percentage(wrong_identity, len(enrolled)),
        "unknown_person_false_match_percent": percentage(false_matches, len(unenrolled)),
        "zone_entry_recall_percent": percentage(detected_entries, len(expected_entries)),
        "false_zone_entry_percent": percentage(false_zone_entries, len(expected_non_entries)),
        "presentation_attack_acceptance_percent": percentage(accepted_attacks, len(attack_trials)),
        "alerts_per_trial_percent": percentage(alerts, len(rows)),
        "mean_entry_to_alert_seconds": round(mean(latencies), 3) if latencies else "Not measured",
        "mean_fps": round(mean(frame_rates), 2) if frame_rates else "Not measured",
    }

def print_summary(rows):
    results = summarize(rows)
    for name, value in results.items():
        print(f"{name.replace('_', ' ').title()}: {value}")

if trials:
    print_summary(trials)
else:
    print("No success rate can be calculated until I enter measured trials.")

## 4. Compare my scenarios

I keep the scenario names consistent. Twins must be enrolled as separate identities, and I test each twin several times in each condition. For printed photos and screens I record what happened, but a correct face match does not mean the system detected a spoof.

In [ ]:
by_scenario = defaultdict(list)
for trial in trials:
    by_scenario[trial["scenario"]].append(trial)

if not by_scenario:
    print("No scenarios to compare yet.")
else:
    for scenario, rows in sorted(by_scenario.items()):
        print(f"\n--- {scenario} ({len(rows)} trial(s)) ---")
        print_summary(rows)

## 5. Scenarios I should test

| Scenario name | How I test it | What I record |
|---|---|---|
| `normal_light` | Enrolled and unenrolled people in steady room light | Identity result, zone entry, alert time, FPS |
| `low_light` | Repeat the same people and camera position in dim light | Same measures and whether the face was too unclear |
| `cap` | Repeat with a cap; keep face visible and then partially covered | Correct identity, uncertain, false reject |
| `twins` | Enroll twins under separate labels and test each many times | Which twin was expected and which identity was returned |
| `printed_photo` | Show a consenting participant's printed photo to the camera | Whether a face match occurred; this is a spoof test observation only |
| `screen_photo` | Show a still face photo on a phone or monitor | Same; do not describe a match as liveness detection |
| `video_replay` | Play a short face video on a screen | Same, plus any obvious replay behavior |
| `camera_disconnect` | Disconnect the camera during a controlled run | Time to dashboard warning and whether monitoring stops safely |

I also note the model file names, code commit, camera and resolution, lighting description, zone shape, face threshold, computer hardware, and number of people/trials. I use separate people or captures for tuning and final evaluation.

## 6. Save a shareable summary

This saves only aggregate metrics, not names or images. I should review the output before sharing it.

In [ ]:
import csv
from pathlib import Path

if trials:
    output_path = Path("scenario_summary.csv")
    summary_rows = []
    for scenario, rows in sorted(by_scenario.items()):
        summary_rows.append({"scenario": scenario, **summarize(rows)})
    with output_path.open("w", newline="", encoding="utf-8") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=summary_rows[0].keys())
        writer.writeheader()
        writer.writerows(summary_rows)
    print(f"I saved aggregate results to: {output_path.resolve()}")
else:
    print("I have no measured results to save yet.")